ARTI308 - Machine Learning

# Lab 4: Data Quality Assessment and Preprocessing

In real-world machine learning projects, data is often:
- Incomplete (missing values)
- Noisy (outliers or random errors)
- Inconsistent (wrong formats, mixed units)

Before building any machine learning model, we must clean and prepare the data properly.

![step2.png](img/step2.png)

In this lab, we will apply practical preprocessing techniques step by step using the Heart Disease dataset.

- **Problem Type:** Binary Classification
- **Target Variable:** `target` (0 = No Disease, 1 = Disease)
- **Dataset Source:** [UCI Heart Disease](https://archive.ics.uci.edu/ml/datasets/heart+Disease)

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

## 1. Load Dataset

In [ ]:
df = pd.read_csv("heart.csv")
df.head(10)

## 2. Data Quality Assessment
### 2.1 Check Data Types
Data types must match the real meaning of each column.
For example:
- `age`, `chol`, `trestbps` should be numeric
- `sex`, `cp`, `fbs` are categorical but may be stored as integers

In [ ]:
df.dtypes

We observe that several columns are stored as `float64` even though they represent categories.

Columns like `sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, and `thal` are categorical variables encoded as integers.

This is a data quality issue -- incorrect types prevent proper analysis and can cause models to treat category codes as continuous numerical values.

### 2.2 Convert Incorrect Data Types
We will convert categorical columns stored as integers to the `category` type for clarity.

In [ ]:
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

for col in categorical_cols:
    df[col] = df[col].astype('category')

df.dtypes

Now the categorical columns are properly typed as `category`.

This helps with correct grouping, plotting, and prevents treating category codes as continuous values in downstream analysis.

## 3. Handling Missing Values
### 3.1 Detect Missing Values
Missing values reduce data quality and can affect model performance.

In [ ]:
df.isna().sum()

The output shows whether any column contains missing values.
If all values are zero, the dataset is complete.
The Heart Disease dataset has no missing values.

### 3.2 Demonstration: Introduce Artificial Missing Values
### Why?

Since our dataset has no missing values, we introduce artificial ones *for learning purposes*.

We will be running this line:

`df_missing.loc[0:5, 'chol'] = np.nan`

- `df_missing`: The pandas DataFrame you are modifying.
- `.loc[0:5, 'chol']`: This uses the label-based indexer to select specific rows and columns.
  - `0:5`: Selects rows with index labels 0, 1, 2, 3, 4, and 5. In label-based indexing, the end index is inclusive.
  - `'chol'`: Selects the column named 'chol'.
- `= np.nan`: Assigns the value `np.nan` (which stands for "Not a Number") to all the selected cells. This is the standard way to represent missing or null values in numerical columns in pandas.

In [ ]:
df_missing = df.copy()
# Convert chol back to float to allow NaN
df_missing['chol'] = df_missing['chol'].astype('float64')
df_missing.loc[0:5, 'chol'] = np.nan
df_missing.isna().sum()

Now the `chol` column contains 6 missing values.

In [ ]:
print("Original shape:", df.shape)
print("After introducing missing values:", df_missing.shape)
df_missing.head(10)

### Strategy 1: Remove Records
This strategy removes records containing missing data.
It works well if the number of missing rows is small.

In [ ]:
df_removed = df_missing.dropna()
print("Shape after removal:", df_removed.shape)
df_removed.isna().sum()

The dataset now has fewer rows.
If only a small portion of data was missing, this method is acceptable.

However, removing too many rows can reduce model performance.

### Strategy 2: Mean Imputation

![Mean.png](img/Mean.png)

The mean represents the average value.
It is commonly used for normally distributed data.

In [ ]:
df_imputed_mean = df_missing.copy()
df_imputed_mean['chol'].fillna(df_imputed_mean['chol'].mean(), inplace=True)

print("Missing values after mean imputation:")
print(df_imputed_mean.isna().sum())
df_imputed_mean.head(10)

Missing values are now replaced with the average cholesterol value.
This preserves dataset size but may reduce variability.
Mean imputation is sensitive to outliers.

### Strategy 3: Median Imputation

![median_formula_2.png](img/median_formula_2.png)

The median is more robust to outliers than the mean.
It is preferred for skewed data.

In [ ]:
df_imputed_median = df_missing.copy()
df_imputed_median['chol'].fillna(df_imputed_median['chol'].median(), inplace=True)

print("Missing values after median imputation:")
print(df_imputed_median.isna().sum())
df_imputed_median.head(10)

Missing values are replaced with the middle value.
This approach is safer when data contains extreme values.

## 4. Handling Outliers
Outliers are extreme values that can distort models.
We will detect outliers using the IQR method.

![IQR.png](img/IQR.png)

In [ ]:
# Reset df to use original numeric types for outlier analysis
df_numeric = pd.read_csv("heart.csv")

plt.figure(figsize=(6, 4))
sns.boxplot(x=df_numeric['chol'])
plt.title("Boxplot of Cholesterol")
plt.show()

Points outside the whiskers represent potential outliers.
High cholesterol values may be valid medical readings or data errors.

### Detect Outliers using IQR
**Method: Interquartile Range (IQR)**

The IQR method defines outliers as values outside:

`Q1 - 1.5 * IQR`  and  `Q3 + 1.5 * IQR`

In [ ]:
Q1 = df_numeric['chol'].quantile(0.25)
Q3 = df_numeric['chol'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower}, Upper bound: {upper}")

outliers = df_numeric[(df_numeric['chol'] < lower) | (df_numeric['chol'] > upper)]
print(f"\nNumber of outliers: {len(outliers)}")
outliers.head(15)

The output displays records considered extreme based on statistical boundaries.
These are patients with unusually high or low cholesterol values.

### Remove Outliers
We remove values outside the acceptable range.

In [ ]:
df_no_outliers = df_numeric[(df_numeric['chol'] >= lower) & (df_numeric['chol'] <= upper)]
print("Original shape:", df_numeric.shape)
print("After removing outliers:", df_no_outliers.shape)

The dataset size is slightly reduced.
Removing outliers reduces distortion but may also remove important rare events.

#### Important Note on Removing Outliers

Not all outliers are errors.

Some extreme values may represent rare but important real-world observations.
For example, in medical data, a very high cholesterol reading might correspond to a patient with a serious condition that is clinically significant.

If we remove such values blindly, we may lose valuable information and bias the analysis.

Before removing outliers, we should always ask:
- Is this value a data entry mistake?
- Or is it a valid but rare observation?

### Capping Outliers (Percentile Method)
Instead of removing outliers, we replace extreme values with percentile limits.

![percentile.png](img/percentile.png)

In [ ]:
lower_cap = df_numeric['chol'].quantile(0.05)
upper_cap = df_numeric['chol'].quantile(0.95)

df_capped = df_numeric.copy()
df_capped['chol'] = df_capped['chol'].clip(lower_cap, upper_cap)

## 5. Data Transformation -- Normalization
Normalization scales numerical features to a similar range.
This ensures that no feature influences the model simply because it has larger numerical values.

### Min-Max Normalization
Min-Max normalization rescales numerical values to a fixed range, usually between 0 and 1.

It works using the formula:
![min_max.png](img/min_max.png)

This method preserves the original distribution shape and relative ordering of values.

Min-Max normalization is especially useful for distance-based models such as:
- K-Nearest Neighbors (KNN)
- K-Means clustering
- Support Vector Machines (SVM)

These models rely on distance calculations, and if features are on very different scales, one feature can dominate the distance computation.

In [ ]:
df_numeric[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']].head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = df_numeric[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']].copy()
df_scaled[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']] = scaler.fit_transform(df_scaled)
df_scaled.head()

After applying Min-Max normalization, all numerical values are scaled to the range between 0 and 1.

The smallest value in each feature becomes 0, and the largest becomes 1.
All other values are proportionally mapped between these two limits.

Importantly, normalization does NOT change the relative relationships between data points.
If one patient originally had higher cholesterol than another, they will still have a higher normalized value.

### Z-Score Normalization
Z-score standardization transforms the data so that:

- The mean of each feature becomes 0
- The standard deviation becomes 1

This is done by subtracting the mean and dividing by the standard deviation:

![zscore.png](img/zscore.png)

This method keeps the shape of the distribution but rescales it around zero.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_standardized = df_numeric[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']].copy()
df_standardized[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']] = scaler.fit_transform(df_standardized)
df_standardized.head()

After standardization, the numerical features are centered around 0.
Values above the original mean become **positive**, and values below the mean become **negative**.

The standard deviation of each feature becomes approximately 1, meaning the spread of the data is standardized.

This transformation is especially useful for:
- Linear regression
- Support Vector Machines (SVM)
- PCA

Because these models assume features are centered and scaled similarly.

## Check Correlation Before Applying PCA

We will check whether numerical features are correlated. If features are strongly correlated, they contain overlapping information.

- **Correlation close to 1** --> Strong positive linear relationship
  (As one feature increases, the other also increases.)

- **Correlation close to -1** --> Strong negative linear relationship
  (As one feature increases, the other decreases.)

- **Correlation close to 0** --> Weak or no linear relationship
  (The features do not move together in a predictable linear way.)

In such cases, dimensionality reduction using PCA is meaningful
because we can combine correlated features into fewer components.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_standardized.corr(), annot=True, cmap="coolwarm", fmt='.2f')
plt.title("Correlation Heatmap (Before PCA)")
plt.show()

The heatmap shows the correlation between the numerical features of the Heart Disease dataset.

- The diagonal values are 1 because each feature is perfectly correlated with itself.
- Notable correlations may appear between features such as `age` and `thalach` (max heart rate tends to decrease with age).
- Features with moderate correlation suggest overlapping information that PCA can compress.

When features show meaningful correlations, PCA can effectively reduce dimensionality while preserving most of the variance.

## 6. Data Reduction -- Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is a dimensionality reduction technique.

Instead of working directly with the original features, PCA creates new features called **principal components**.

These components:

- Are linear combinations of the original features
- Are uncorrelated with each other
- Capture variance in descending order (from most important to least)

The first principal component (PC1) captures the largest possible variance in the dataset.

The second principal component (PC2) captures the next largest variance, while being orthogonal (perpendicular) to PC1.

This allows us to reduce dimensionality while retaining most of the important information in the data.

### Visual Intuition

Imagine we have two features:

X1 = Age
X2 = Max Heart Rate

If we plot the data points, they may look like this:

```
              X2
               |
               |
               |        *
               |      *
               |    *
               |  *
               | *
               |________________________ X1
```

Notice that the points follow a diagonal pattern.
This means the two features are correlated and contain overlapping information.

Instead of keeping both X1 and X2 separately,
PCA finds the direction where the data varies the most.

That direction becomes **Principal Component 1 (PC1)**.

```
              X2
               |
               |        *
               |      *
               |    *
               |  *
               | *
               |________________________ X1
                    \
                     \
                      \
                       \
                        -> PC1 (maximum variance direction)
```

PC2 is the direction perpendicular to PC1.

If most of the variation is along PC1,
then PC1 alone captures most of the dataset's information.

In that case, we can reduce:

5 features --> 2 features (PC1 + PC2)

while keeping most of the variance.

In [ ]:
from sklearn.decomposition import PCA

X = df_standardized[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']]

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Total Variance Explained:", sum(pca.explained_variance_ratio_))

The `Explained Variance Ratio` indicates how much of the total information (variance) is captured by each principal component.

For example:
- If PC1 explains 40% and PC2 explains 25% of the variance, then together they summarize 65% of the dataset's information using only 2 features instead of 5.
- The higher the total explained variance, the more effective the dimensionality reduction.

When most of the variance is captured by fewer components, dimensionality reduction is considered effective.

This helps simplify models, reduce computational cost, and sometimes improve generalization performance.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(principal_components[:, 0], principal_components[:, 1], 
            c=df_numeric['target'], cmap='coolwarm', alpha=0.6)
plt.colorbar(label='Heart Disease (0=No, 1=Yes)')
plt.title("PCA Projection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

Each point in this plot represents one patient.

The axes no longer represent the original features (age, cholesterol, etc.).
Instead:

- The horizontal axis represents Principal Component 1 (PC1).
- The vertical axis represents Principal Component 2 (PC2).

The points are colored by the `target` variable:
- Blue points represent patients without heart disease (target = 0).
- Red points represent patients with heart disease (target = 1).

If the two classes show some separation in this reduced space, it suggests that the original features contain useful discriminative information for classification.

# Assignment

In this assignment, you will:
- **Task 1:** Identify data quality issues in the dataset.
- **Task 2:** Apply one missing value strategy and explain why.
- **Task 3:** Detect and handle outliers using IQR.
- **Task 4:** Normalize numerical features using both Min-Max and Z-score.
- **Task 5:** Apply PCA and interpret explained variance.

End of Lab 4.